# GPU Released Memory Failure — yhlleo/mnist

**Smell (`mnist_deep.py` line 8):** `tf.InteractiveSession()` is opened at the top of the script and is **never closed**. When the script is run repeatedly (10 runs), each session occupies GPU memory that is never freed.

**Fix:** Close the session with `sess.close()` and call `tf.compat.v1.reset_default_graph()` after each run.

In [ ]:
!pip install -q codecarbon

In [ ]:
import codecarbon, os, json
_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_dir, exist_ok=True)
_p = os.path.join(_dir, 'nordic_emissions.json')
if not os.path.exists(_p):
    with open(_p, 'w') as f: json.dump({}, f)
print('CodeCarbon ready')

In [ ]:
import gc, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.compat.v1.disable_eager_execution()
from codecarbon import EmissionsTracker

os.makedirs('results', exist_ok=True)
N_RUNS  = 10
N_STEPS = 2000   # reduced from 20 000 for Kaggle feasibility

# Load MNIST using TF1-style helper
from tensorflow.keras.datasets import mnist as _mnist
(x_tr, y_tr), (x_te, y_te) = _mnist.load_data()
x_tr = x_tr.reshape(-1, 784).astype('float32') / 255.0
x_te = x_te.reshape(-1, 784).astype('float32') / 255.0
from tensorflow.keras.utils import to_categorical
y_tr_oh = to_categorical(y_tr, 10)
y_te_oh = to_categorical(y_te, 10)

def next_batch(x, y, batch_size=50):
    idx = np.random.choice(len(x), batch_size, replace=False)
    return x[idx], y[idx]

print('MNIST loaded')

In [ ]:
def build_graph():
    """Rebuild the TF1 graph from yhlleo/mnist/mnist_deep.py"""
    def weight_variable(shape):
        return tf.compat.v1.Variable(tf.compat.v1.truncated_normal(shape, stddev=0.1))
    def bias_variable(shape):
        return tf.compat.v1.Variable(tf.constant(0.1, shape=shape))
    def conv2d(x, W):
        return tf.nn.conv2d(x, W, strides=[1,1,1,1], padding='SAME')
    def max_pool_2x2(x):
        return tf.nn.max_pool(x, ksize=[1,2,2,1], strides=[1,2,2,1], padding='SAME')

    x  = tf.compat.v1.placeholder('float', [None, 784])
    y_ = tf.compat.v1.placeholder('float', [None, 10])
    keep_prob = tf.compat.v1.placeholder('float')

    w_conv1 = weight_variable([5,5,1,32]); b_conv1 = bias_variable([32])
    x_image = tf.reshape(x, [-1,28,28,1])
    h_pool1 = max_pool_2x2(tf.nn.relu(conv2d(x_image, w_conv1) + b_conv1))

    w_conv2 = weight_variable([5,5,32,64]); b_conv2 = bias_variable([64])
    h_pool2 = max_pool_2x2(tf.nn.relu(conv2d(h_pool1, w_conv2) + b_conv2))

    w_fc1 = weight_variable([7*7*64, 1024]); b_fc1 = bias_variable([1024])
    h_fc1 = tf.nn.relu(tf.matmul(tf.reshape(h_pool2, [-1,7*7*64]), w_fc1) + b_fc1)
    h_fc1_drop = tf.nn.dropout(h_fc1, rate=1-keep_prob)

    w_fc2 = weight_variable([1024, 10]); b_fc2 = bias_variable([10])
    y_conv = tf.nn.softmax(tf.matmul(h_fc1_drop, w_fc2) + b_fc2)

    cross_entropy = -tf.reduce_sum(y_ * tf.math.log(y_conv + 1e-8))
    train_step = tf.compat.v1.train.GradientDescentOptimizer(1e-3).minimize(cross_entropy)
    correct = tf.equal(tf.argmax(y_conv,1), tf.argmax(y_,1))
    accuracy = tf.reduce_mean(tf.cast(correct, 'float'))
    return x, y_, keep_prob, train_step, accuracy

print('Graph builder ready')

## BEFORE — Smell Active (InteractiveSession never closed)

In [ ]:
results_before = []

for run in range(1, N_RUNS + 1):
    print(f'\n[BEFORE] Run {run}/{N_RUNS}')
    # ❌ SMELL: new session opened but never closed — mirrors yhlleo/mnist/mnist_deep.py
    sess = tf.compat.v1.InteractiveSession()
    x, y_, keep_prob, train_step, accuracy = build_graph()
    sess.run(tf.compat.v1.global_variables_initializer())

    tracker = EmissionsTracker(
        project_name=f'yhlleo_mnist_before_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    for i in range(N_STEPS):
        bx, by = next_batch(x_tr, y_tr_oh)
        sess.run(train_step, feed_dict={x: bx, y_: by, keep_prob: 0.5})
    acc = sess.run(accuracy, feed_dict={x: x_te, y_: y_te_oh, keep_prob: 1.0})
    tracker.stop()
    em = tracker.final_emissions_data
    results_before.append({
        'run': run, 'accuracy': round(float(acc), 4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    # ❌ SMELL: sess.close() intentionally omitted — GPU memory not freed

df_before = pd.DataFrame(results_before)
df_before.to_csv('results/yhlleo_mnist_before.csv', index=False)
print('\n--- BEFORE means ---')
print(df_before.mean(numeric_only=True))
print('Saved results/yhlleo_mnist_before.csv')

## AFTER — Smell Fixed (sess.close() + reset_default_graph between runs)

In [ ]:
results_after = []

for run in range(1, N_RUNS + 1):
    print(f'\n[AFTER] Run {run}/{N_RUNS}')
    # ✅ FIX: reset graph so no stale nodes from previous run
    tf.compat.v1.reset_default_graph()
    sess = tf.compat.v1.InteractiveSession()
    x, y_, keep_prob, train_step, accuracy = build_graph()
    sess.run(tf.compat.v1.global_variables_initializer())

    tracker = EmissionsTracker(
        project_name=f'yhlleo_mnist_after_run{run}',
        save_to_file=False, log_level='error'
    )
    tracker.start()
    for i in range(N_STEPS):
        bx, by = next_batch(x_tr, y_tr_oh)
        sess.run(train_step, feed_dict={x: bx, y_: by, keep_prob: 0.5})
    acc = sess.run(accuracy, feed_dict={x: x_te, y_: y_te_oh, keep_prob: 1.0})
    tracker.stop()
    em = tracker.final_emissions_data
    results_after.append({
        'run': run, 'accuracy': round(float(acc), 4),
        'energy_kWh': round(em.energy_consumed,8),
        'cpu_energy_kWh': round(em.cpu_energy,8),
        'gpu_energy_kWh': round(em.gpu_energy,8),
        'ram_energy_kWh': round(em.ram_energy,8),
        'emissions_kgCO2': round(em.emissions,8),
        'duration_s': round(em.duration,2),
    })
    print(f'  acc={acc:.4f}  energy={em.energy_consumed:.6f} kWh  CO2={em.emissions:.6f} kg  t={em.duration:.1f}s')
    # ✅ FIX: close session to release GPU memory
    sess.close()
    gc.collect()

df_after = pd.DataFrame(results_after)
df_after.to_csv('results/yhlleo_mnist_after.csv', index=False)
print('\n--- AFTER means ---')
print(df_after.mean(numeric_only=True))
print('Saved results/yhlleo_mnist_after.csv')

In [ ]:
print('\n=== SUMMARY: yhlleo/mnist ===')
print(f"BEFORE avg energy : {df_before['energy_kWh'].mean():.6f} kWh")
print(f"AFTER  avg energy : {df_after['energy_kWh'].mean():.6f} kWh")
print(f"BEFORE avg CO2    : {df_before['emissions_kgCO2'].mean():.6f} kg")
print(f"AFTER  avg CO2    : {df_after['emissions_kgCO2'].mean():.6f} kg")
print(f"BEFORE avg time   : {df_before['duration_s'].mean():.1f} s")
print(f"AFTER  avg time   : {df_after['duration_s'].mean():.1f} s")